In [1]:
import os
import sys

for _ in range(2):
    os.chdir("..")

os.getcwd()

'/Users/a.komnatskii/Documents/mkn/inf_search/six-00-twenty-search'

In [2]:
import dev.make_dataset

In [3]:
from data.data_loader import load_documents

from dev.index import *
from dev.query_parser import *
from dev.posting_list import BasePostingList, PostingList, AntiPostingList
from dev.search_engine import SearchEngine

In [4]:
from importlib import reload

import dev.search_engine
reload(dev.search_engine)

<module 'dev.search_engine' from '/Users/a.komnatskii/Documents/mkn/inf_search/six-00-twenty-search/dev/search_engine.py'>

In [5]:
dev.search_engine?

Type:        module
String form: <module 'dev.search_engine' from '/Users/a.komnatskii/Documents/mkn/inf_search/six-00-twenty-search/dev/search_engine.py'>
File:        ~/Documents/mkn/inf_search/six-00-twenty-search/dev/search_engine.py
Docstring:   <no docstring>

# Search Engine API

In [6]:
SearchEngine?

Init signature:
SearchEngine(
    reverse_index_paths={'text_file_path': 'data/wikipedia_100k_delta_index_text.pb', 'title_file_path': 'data/wikipedia_100k_delta_index_title.pb'},
    pos_index_paths={'text_file_path': 'data/wikipedia_100k_pos_index_text.pb', 'title_file_path': 'data/wikipedia_100k_pos_index_title.pb'},
    parser=None,
)
Docstring:      Search engine with boolean queries
File:           ~/Documents/mkn/inf_search/six-00-twenty-search/dev/search_engine.py
Type:           type
Subclasses:     

In [7]:
engine = SearchEngine()
load_documents(engine, 'data/documents_100k')

✅ Loaded 150000 documents from data/documents_100k


# Test Cases

In [8]:
q = 'Восточной'

In [9]:
engine.documents.get(0).keys()

dict_keys(['doc_id', 'text', 'title', 'url'])

In [10]:
res1 = engine.search('Европе')
res1[:10]

[('0', 0.0),
 ('1', 0.0),
 ('3', 0.0),
 ('5', 0.0),
 ('6', 0.0),
 ('8', 0.0),
 ('9', 0.0),
 ('10', 0.0),
 ('11', 0.0),
 ('16', 0.0)]

In [11]:
res2 = engine.search('Восточной')
res2[:10]

[('0', 0.0),
 ('1', 0.0),
 ('2', 0.0),
 ('3', 0.0),
 ('5', 0.0),
 ('9', 0.0),
 ('10', 0.0),
 ('11', 0.0),
 ('12', 0.0),
 ('16', 0.0)]

In [12]:
parser = QueryParser()

In [13]:
ast = parser.parse('Восточной OR Европе')
ast

OR(Term(text:восточной), Term(text:европе))

In [14]:
engine.search('Восточной OR Европе')[:10]

[('0', 0.0),
 ('1', 0.0),
 ('2', 0.0),
 ('3', 0.0),
 ('5', 0.0),
 ('6', 0.0),
 ('8', 0.0),
 ('9', 0.0),
 ('10', 0.0),
 ('11', 0.0)]

In [15]:
ast = parser.parse('Восточной Европе @2')
ast

Near(Term(text:восточной) Term(text:европе), k=2)

In [16]:
type(ast), ast.terms, ast.k

(dev.query_parser.NearNode, [Term(text:восточной), Term(text:европе)], 2)

# Пример

In [17]:
%%time
engine.search('Северной Европе @2')[:10]

CPU times: user 20.4 ms, sys: 27.5 ms, total: 47.9 ms
Wall time: 47.5 ms


[('0', 0.0),
 ('11', 0.0),
 ('36', 0.0),
 ('52', 0.0),
 ('57', 0.0),
 ('63', 0.0),
 ('74', 0.0),
 ('117', 0.0),
 ('311', 0.0),
 ('409', 0.0)]

In [18]:
engine.documents.get(0)

{'doc_id': 0,
 'text': 'Литва́ ( ), официальное название\xa0— Лито́вская Респу́блика ()\xa0— государство, расположенное в Северной Европе. Площадь\xa0—  км². Протяжённость с севера на юг\xa0— 280\xa0км, а с запада на восток\xa0— 370 км. Население составляет  человек (август, 2023). Занимает 137-е место в мире по численности населения и 121-е по территории. Имеет выход к Балтийскому морю, расположена на его восточном побережье. Береговая линия составляет всего 99 км (наименьший показатель среди государств Балтии). На севере граничит с Латвией, на юго-востоке\xa0— с Белоруссией, на юго-западе\xa0— с Польшей и Калининградской областью России. По площади и населению является самым крупным государством из стран Балтии.\n\nСтолица\xa0— Вильнюс. Официальный язык\xa0— литовский. Денежная единица\xa0— евро.\n\nВосстановление независимости страны провозглашено 11 марта 1990\xa0года. 6 сентября 1991\xa0года Государственный совет СССР признал независимость Литвы. \n\nЛитва\xa0— член ООН (1991), ОБ

In [19]:
len(engine.search('Северной AND Европе'))

2465

In [20]:
engine.pos_indexer.get(0, 'северной')

[19]

In [21]:
engine.pos_indexer.get(0, 'европе')

[20, 3683, 5540, 5673, 5907, 6094]

In [22]:
 engine.near_in(0, ['северной', 'европе'])

True

In [23]:
for doc_id, _ in engine.search('Северной AND Европе'):
    
    print(doc_id, ':', engine.near_in(int(doc_id), ['северной', 'европе']))

0 : True
1 : False
3 : False
8 : False
9 : False
10 : False
11 : True
16 : False
17 : False
19 : False
21 : False
22 : False
25 : False
26 : False
27 : False
36 : True
39 : False
40 : False
41 : False
45 : False
46 : False
49 : False
52 : True
57 : True
60 : False
62 : False
63 : True
65 : False
68 : False
74 : True
75 : False
76 : False
77 : False
83 : False
117 : True
119 : False
120 : False
125 : False
162 : False
177 : False
194 : False
197 : False
198 : False
262 : False
263 : False
265 : False
266 : False
267 : False
304 : False
311 : True
341 : False
343 : False
356 : False
368 : False
369 : False
386 : False
395 : False
397 : False
405 : False
409 : True
410 : True
411 : True
412 : False
413 : False
414 : False
422 : False
423 : False
424 : True
439 : False
445 : False
479 : False
502 : False
565 : True
571 : False
573 : True
582 : False
595 : False
614 : True
621 : False
625 : False
633 : False
635 : False
637 : False
652 : False
663 : False
787 : False
790 : False
791 : False